In [1]:
import pandas as pd
import numpy as np


# Generando un sistema coherente que permita calcular KPIs reales

## Goals:

1. Generar empleados
150–300
con regiones, fechas, costos
2. Generar timeline (12 meses)
3. Generar horas por mes
coherentes con disponibilidad
4. Generar asignaciones
con porcentajes realistas
5. Generar costos
consistentes con salario
6. Generar objetivos
metas razonables

Genera TODAS las tablas en Python

Y exporta:

1. empleados.csv
2. horas_trabajadas.csv
3. asignaciones.csv
4. costos.csv
5. objetivos.csv
6. calendario.csv

TABLA calendario
¿Por qué es clave?

**Porque TODO depende del tiempo:
KPIs mensuales
filtros
comparaciones**


# **KPIs (definición tipo empresa real)**


***1. Headcount***

 Número de empleados activos en un periodo

Empleado activo si:
fecha_ingreso <= fin_mes
y (fecha_salida es null o > inicio_mes)

***2. Turnover (Rotación)***

Qué tan rápido pierdes empleados

Turnover mensual =
empleados que salieron en el mes
/
headcount promedio del mes

***3.Utilización de capacidad***

Qué tanto estás usando tu workforce

- Utilización_real =
horas_trabajadas / horas_disponibles

***4.Utilización facturable***

Qué tanto genera dinero

- Utilización facturable =
horas_facturables / horas_disponibles

***5.Costos***

Cuánto cuesta operar

- Costo total =
SUM(costo_laboral)

***6.Cumplimiento de objetivos***

Ejemplo:

Cumplimiento =
valor_real / objetivo

- Cumplimiento =
(SUM(horas_trabajadas) / SUM(horas_disponibles))
/
objetivo_utilizacion

**7. Desviación**

- desviación = (costo_real - objetivo_costo) / objetivo_costo




**TODOS estos KPIs dependen de:
tiempo, (mes)
empleado
región**

El estado depende del periodo

Empleado activo en un mes si:
fecha_ingreso <= fin_mes
y (fecha_salida es null o > inicio_mes)

In [2]:
import pandas as pd
import numpy as np

np.random.seed(42)

n_empleados = 200

# IDs
ids = [f"E{str(i).zfill(3)}" for i in range(1, n_empleados+1)]

# Catálogos
regiones = ["LATAM", "North America", "Europe", "APAC"]
areas = ["Operations", "HR", "Finance", "IT"]
puestos = ["Analyst", "Senior Analyst", "Manager", "Director"]

# Base
empleados = pd.DataFrame({
    "id_empleado": ids,
    "region": np.random.choice(regiones, n_empleados),
    "area": np.random.choice(areas, n_empleados),
    "puesto": np.random.choice(puestos, n_empleados),
})

**fecha_ingreso: 2023–2025 (para tener históricos)
periodos: SOLO 2025**

In [3]:
# Fecha ingreso (histórico realista)
empleados["fecha_ingreso"] = pd.to_datetime(
    np.random.choice(
        pd.date_range("2023-01-01", "2025-09-01"),
        n_empleados
    )
)

**4. fecha de salida (simular rotación)**

In [4]:
fechas_salida = []

for ingreso in empleados["fecha_ingreso"]:

    # Probabilidad de salida
    if np.random.rand() < 0.25:  # 25% rotación

        salida = ingreso + pd.Timedelta(days=np.random.randint(90, 500))

        # Limitar al 2025 (porque nuestro análisis es 2025)
        if salida > pd.Timestamp("2025-12-31"):
            salida = pd.NaT

    else:
        salida = pd.NaT

    fechas_salida.append(salida)

empleados["fecha_salida"] = fechas_salida

costo_laboral = horas_trabajadas * costo_hora

**Paso 5: costos y capacidad**

In [5]:
empleados["costo_hora"] = np.random.uniform(12, 45, n_empleados).round(2)
empleados["horas_disponibles_mes"] = 160

**6. país**

In [6]:
# ya hay empleados con region, pero no por país

paises = {
    "LATAM": ["México", "Argentina", "Chile"],
    "North America": ["USA", "Canada"],
    "Europe": ["Spain", "Germany", "UK"],
    "APAC": ["India", "Japan", "Australia"]
}

empleados["pais"] = empleados["region"].apply(
    lambda r: np.random.choice(paises[r])
)

In [7]:
empleados.head()
empleados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id_empleado            200 non-null    object        
 1   region                 200 non-null    object        
 2   area                   200 non-null    object        
 3   puesto                 200 non-null    object        
 4   fecha_ingreso          200 non-null    datetime64[ns]
 5   fecha_salida           43 non-null     datetime64[ns]
 6   costo_hora             200 non-null    float64       
 7   horas_disponibles_mes  200 non-null    int64         
 8   pais                   200 non-null    object        
dtypes: datetime64[ns](2), float64(1), int64(1), object(5)
memory usage: 14.2+ KB


1. 200 empleados ✔
2. fecha_salida: 43 non-null → ~21.5% rotación ✔ (coherente)
3. Tipos correctos (datetime, float, etc.) ✔
4. No hay nulos raros ✔
5. pais consistente ✔

Si alguien tiene:

160 horas disponibles

Entonces:

Escenario	Horas trabajadas	Utilización
1. Baja	80	50%
2. Media	120–130	75–80%
3. Alta	150–160	95–100%
4. Sobrecarga	170+	>100%

***El estado no se guadó porque el estado es dependiente del tiempo, y debe calcularse dinámicamente por periodo para evitar inconsistencias en el análisis.***

# **FASE 3 — Generar horas_trabajadas**

Sin calendario pasa esto:

1. ❌ No hay eje temporal
2. ❌ No puedes generar horas por mes
3. ❌ No puedes calcular KPIs

In [8]:
calendario = pd.DataFrame({
    "periodo": pd.date_range("2025-01-01", "2025-12-01", freq="MS")
})

In [9]:
calendario["anio"] = calendario["periodo"].dt.year
calendario["mes"] = calendario["periodo"].dt.month
#calendario["periodo_str"] = calendario["periodo"].dt.strftime("%Y-%m")
calendario["periodo"] = calendario["periodo"].dt.strftime("%Y-%m") # consistencia, así todas las tablas usan el mismo formato

Estructura

Cada fila será:

| id_empleado | periodo | horas_trabajadas | horas_facturables | horas_extra |

Lógica clave

Para cada:

1. empleado
2. mes

Hacemos:

1. SI está activo → genera horas
2. SI NO → no genera nada

In [10]:
horas_data = []

for _, emp in empleados.iterrows():

    for periodo in calendario["periodo"]:

        inicio_mes = periodo
        fin_mes = periodo + pd.offsets.MonthEnd(0)

        # verificar si está activo
        activo = (
            (emp["fecha_ingreso"] <= fin_mes) and
            (pd.isna(emp["fecha_salida"]) or emp["fecha_salida"] > inicio_mes)
        )

        if not activo:
            continue

        # definir nivel de utilización
        nivel = np.random.choice(["baja", "media", "alta"], p=[0.2, 0.6, 0.2])

        if nivel == "baja":
            horas = np.random.randint(80, 110)
        elif nivel == "media":
            horas = np.random.randint(120, 140)
        else:
            horas = np.random.randint(145, 165)

        # facturable vs no facturable
        facturable = int(horas * np.random.uniform(0.6, 0.9))
        no_facturable = horas - facturable

        # overtime
        extra = max(0, horas - 160)

        horas_data.append({
            "id_empleado": emp["id_empleado"],
            "periodo": periodo.strftime("%Y-%m"),
            "horas_trabajadas": horas,
            "horas_facturables": facturable,
            "horas_no_facturables": no_facturable,
            "horas_extra": extra
        })

horas_trabajadas = pd.DataFrame(horas_data)

In [11]:
horas_trabajadas.head()
horas_trabajadas.describe()

,horas_trabajadas,horas_facturables,horas_no_facturables,horas_extra
count,1855.000000,1855.000000,1855.000000,1855.000000
mean,127.499730,94.730997,32.768733,0.082480
std,20.136317,18.776675,12.518116,0.484039
min,80.000000,48.000000,9.000000,0.000000
25%,121.000000,82.000000,22.000000,0.000000
50%,129.000000,95.000000,32.000000,0.000000
75%,138.000000,108.000000,42.000000,0.000000
max,164.000000,146.000000,66.000000,4.000000


In [12]:
horas_trabajadas

,id_empleado,periodo,horas_trabajadas,horas_facturables,horas_no_facturables,horas_extra
0,E001,2025-05,127,92,35,0
1,E001,2025-06,151,118,33,0
2,E001,2025-07,159,140,19,0
3,E001,2025-08,103,68,35,0
4,E001,2025-09,99,64,35,0
...,...,...,...,...,...,...
1850,E199,2025-08,159,119,40,0
1851,E199,2025-09,161,119,42,1
1852,E199,2025-10,133,100,33,0
1853,E199,2025-11,126,88,38,0


## Análisis Exploratorio – Horas de Trabajo

### 1. Interpretación General

- **Número de registros (`count`)**: 1855  
- **Calidad de datos**:  
  - Sin valores nulos (NaNs)  
  - Dataset consistente  

Esto confirma que el dataset está completo y listo para análisis.

---

### 2. Horas Trabajadas

**Estadísticas principales:**

- Media: ≈ 127 horas  
- Mínimo: 80 horas  
- Máximo: 164 horas  

**Interpretación:**

- 80 → baja utilización  
- 127 → utilización media  
- 164 → alta utilización + horas extra  

Los valores están alineados con un escenario laboral realista.

**Distribución (percentiles):**

- 25%: 121 horas  
- 50% (mediana): 129 horas  
- 75%: 137 horas  

Esto indica que la mayoría de los empleados trabaja entre 120 y 140 horas, lo que representa aproximadamente un 75% – 85% de utilización.

---

### 3. Horas Facturables

- Media: ≈ 94 horas  

Relación clave:

94 / 127 ≈ 74%

Interpretación:

- Buen ratio de facturación  
- Coherente con escenarios reales (no todo el tiempo genera ingresos)

---

### 4. Horas No Facturables

- Media: ≈ 32 horas  

Representan actividades como:

- Reuniones  
- Administración  
- Capacitación  

Este componente aporta realismo al dataset.

---

### 5. Horas Extra

- Media: ≈ 0.07 horas  
- Máximo: 4 horas  

Interpretación:

- La mayoría no realiza horas extra  
- Existen casos puntuales  

Distribución consistente con entornos reales.

---

### 6. Validación de Consistencia

- Máximo de horas trabajadas = 164  
- Límite estándar esperado ≈ 160 horas  

Entonces:

164 - 160 = 4

Esto coincide con el máximo de horas extra, confirmando consistencia interna.

---

### 7. Evaluación Global

El dataset presenta:

- Distribución no uniforme  
- Comportamiento no artificial  
- Valores no extremos  
- Coherencia estadística  

Esto lo hace adecuado como dataset simulado realista.

---

### 8. Insight Profesional

Se puede concluir:

"La organización opera con una utilización promedio cercana al 80%, con baja incidencia de horas extra y una distribución equilibrada entre horas facturables y no facturables."

---

### 9. Headcount Mensual

Para calcular el headcount mensual correctamente:

- Se debe utilizar lógica basada en fechas  
- Determinar el estado del empleado por periodo  

No es suficiente contar registros; es necesario considerar la temporalidad.

In [13]:
# Logica:
# costo = horas_trabajadas * costo_hora

In [14]:
costos = horas_trabajadas.merge(
    empleados[["id_empleado", "region", "costo_hora"]],
    on="id_empleado",
    how="left"
)

costos["costo_laboral"] = (
    costos["horas_trabajadas"] * costos["costo_hora"]
).round(2)

costos["tipo_costo"] = "Salario"
costos["moneda"] = "USD"

In [15]:
costos

,id_empleado,periodo,horas_trabajadas,horas_facturables,horas_no_facturables,horas_extra,region,costo_hora,costo_laboral,tipo_costo,moneda
0,E001,2025-05,127,92,35,0,Europe,24.45,3105.15,Salario,USD
1,E001,2025-06,151,118,33,0,Europe,24.45,3691.95,Salario,USD
2,E001,2025-07,159,140,19,0,Europe,24.45,3887.55,Salario,USD
3,E001,2025-08,103,68,35,0,Europe,24.45,2518.35,Salario,USD
4,E001,2025-09,99,64,35,0,Europe,24.45,2420.55,Salario,USD
...,...,...,...,...,...,...,...,...,...,...,...
1850,E199,2025-08,159,119,40,0,LATAM,23.96,3809.64,Salario,USD
1851,E199,2025-09,161,119,42,1,LATAM,23.96,3857.56,Salario,USD
1852,E199,2025-10,133,100,33,0,LATAM,23.96,3186.68,Salario,USD
1853,E199,2025-11,126,88,38,0,LATAM,23.96,3018.96,Salario,USD


Validando

128 horas * 24.45 = 3129.6 ✔

- cálculo correcto
- join correcto
- columnas correctas

¿Por qué costos viene de horas_trabajadas y no de empleados?

  ❌ Si usaras solo empleados

- costo = costo_hora * 160
- Problema:
- todos cuestan lo mismo cada mes ❌
- ignora si trabajaron o no ❌
- no refleja realidad operativa ❌

En cambio con horas_trabajadas
costo = horas_reales * costo_hora

Ahora sí:

- si trabajó poco → cuesta menos ✔
- si trabajó mucho → cuesta más ✔
- si no trabajó → costo = 0 ✔

 Esto conecta directamente con:

- utilización
- productividad
- eficiencia

Esto se llama:

Cost is activity-driven

- El costo depende de la actividad, no solo del recurso


## 🚀 5. Siguiente Paso: Asignaciones

Este componente permite enriquecer el análisis del dataset incorporando la distribución del trabajo por cliente y proyecto.

---

### 🎯 Objetivo

Con esta estructura se habilitan análisis como:

- Análisis por cliente  
- Análisis por proyecto  
- Evaluación de carga de trabajo real  

---

### 🧩 Estructura de la Tabla

| id_empleado | cliente | proyecto | porcentaje_asignacion |
|------------|--------|----------|------------------------|

---

### 🧠 Lógica de Negocio

Cada empleado puede tener:

- Un solo proyecto (escenario simple)  
- Múltiples proyectos (escenario más realista)  

---

### ⚠️ Restricción Clave

La suma de las asignaciones por empleado debe cumplir:

\[
\sum porcentaje\_asignacion \leq 100\%
\]

---

### 💡 Interpretación

- 100% → completamente asignado  
- < 100% → capacidad disponible  
- > 100% → sobreasignación (debe evitarse)  

---

### 🔍 Valor Analítico

Esta estructura permite:

- Detectar sobrecarga de trabajo  
- Identificar capacidad ociosa  
- Analizar distribución de recursos entre clientes  
- Evaluar dependencia de clientes/proyectos  

💥 Este es un paso clave para llevar el dataset a un nivel profesional (tipo consultoría / BI real).

In [16]:
clientes = ["Cliente A", "Cliente B", "Cliente C", "Cliente D"]
proyectos = ["Alpha", "Beta", "Gamma", "Delta", "Omega"]

asignaciones_data = []

for emp_id in empleados["id_empleado"]:

    # cada empleado puede tener 1 o 2 proyectos
    n_asignaciones = np.random.choice([1, 2], p=[0.7, 0.3])
## hacer el dataset más interesante
     ##n_asignaciones = np.random.choice([1, 2, 3], p=[0.5, 0.3, 0.2])
    # genera porcentajes que suman 100
    porcentajes = np.random.dirichlet(np.ones(n_asignaciones)) * 100

    for i in range(n_asignaciones):

        asignaciones_data.append({
            "id_empleado": emp_id,
            "cliente": np.random.choice(clientes),
            "proyecto": np.random.choice(proyectos),
            "porcentaje_asignacion": round(porcentajes[i], 2)
        })

asignaciones = pd.DataFrame(asignaciones_data)

construir asignaciones por mes

Hasta ahora tenemos:

- horas_trabajadas → empleado + mes ✔
- asignaciones → empleado (sin mes) ⚠️

👉 Necesitamos combinarlas.

**⚙️ Paso 1: hacer join**

In [17]:
asignaciones_mes = costos.merge(
    asignaciones,
    on="id_empleado",
    how="left"
)

- Expandir

empleado → empleado + mes + cliente

 Ahora cada fila representa:

- empleado
- mes
- cliente/proyecto

**Paso 2: repartir horas**

In [18]:
asignaciones_mes["horas_cliente"] = (
    asignaciones_mes["horas_trabajadas"] *
    asignaciones_mes["porcentaje_asignacion"] / 100
).round(2)

**Paso 3: Repartir costos**

In [19]:
asignaciones_mes["costo_cliente"] = (
    asignaciones_mes["costo_laboral"] *
    asignaciones_mes["porcentaje_asignacion"] / 100
).round(2)


**validaciones clave**

In [20]:
asignaciones_mes.groupby(
    ["id_empleado", "periodo"]
)["horas_cliente"].sum()

id_empleado  periodo
E001         2025-05    127.0
             2025-06    151.0
             2025-07    159.0
             2025-08    103.0
             2025-09     99.0
                        ...  
E199         2025-08    159.0
             2025-09    161.0
             2025-10    133.0
             2025-11    126.0
             2025-12    158.0
Name: horas_cliente, Length: 1855, dtype: float64

Validacion extra


In [31]:
check = asignaciones_mes.groupby(
    ["id_empleado", "periodo"]
)["horas_cliente"].sum().reset_index()

check = check.merge(
    horas_trabajadas,
    on=["id_empleado", "periodo"]
)

check["diff"] = (
    check["horas_cliente"] - check["horas_trabajadas"]
)

check["diff"].abs().sum()

np.float64(0.0)

In [21]:
asignaciones_mes.head(10)

,id_empleado,periodo,horas_trabajadas,horas_facturables,horas_no_facturables,horas_extra,region,costo_hora,costo_laboral,tipo_costo,moneda,cliente,proyecto,porcentaje_asignacion,horas_cliente,costo_cliente
0,E001,2025-05,127,92,35,0,Europe,24.45,3105.15,Salario,USD,Cliente D,Omega,100.00,127.00,3105.15
1,E001,2025-06,151,118,33,0,Europe,24.45,3691.95,Salario,USD,Cliente D,Omega,100.00,151.00,3691.95
2,E001,2025-07,159,140,19,0,Europe,24.45,3887.55,Salario,USD,Cliente D,Omega,100.00,159.00,3887.55
3,E001,2025-08,103,68,35,0,Europe,24.45,2518.35,Salario,USD,Cliente D,Omega,100.00,103.00,2518.35
4,E001,2025-09,99,64,35,0,Europe,24.45,2420.55,Salario,USD,Cliente D,Omega,100.00,99.00,2420.55
5,E001,2025-10,161,120,41,1,Europe,24.45,3936.45,Salario,USD,Cliente D,Omega,100.00,161.00,3936.45
6,E001,2025-11,121,95,26,0,Europe,24.45,2958.45,Salario,USD,Cliente D,Omega,100.00,121.00,2958.45
7,E001,2025-12,128,79,49,0,Europe,24.45,3129.60,Salario,USD,Cliente D,Omega,100.00,128.00,3129.60
8,E002,2025-06,130,101,29,0,APAC,12.66,1645.80,Salario,USD,Cliente D,Omega,51.11,66.44,841.17
9,E002,2025-06,130,101,29,0,APAC,12.66,1645.80,Salario,USD,Cliente B,Delta,48.89,63.56,804.63


In [22]:
asignaciones["id_empleado"].value_counts()

,count
id_empleado,
E002,2
E026,2
E023,2
E022,2
E019,2
...,...
E193,1
E196,1
E194,1


In [23]:
asignaciones[asignaciones.duplicated("id_empleado", keep=False)]

,id_empleado,cliente,proyecto,porcentaje_asignacion
1,E002,Cliente D,Omega,51.11
2,E002,Cliente B,Delta,48.89
9,E009,Cliente B,Gamma,6.81
10,E009,Cliente D,Alpha,93.19
11,E010,Cliente C,Omega,53.86
...,...,...,...,...
267,E195,Cliente D,Gamma,95.29
269,E197,Cliente C,Omega,9.62
270,E197,Cliente A,Beta,90.38
273,E200,Cliente D,Beta,72.70


In [24]:
horas_trabajadas

,id_empleado,periodo,horas_trabajadas,horas_facturables,horas_no_facturables,horas_extra
0,E001,2025-05,127,92,35,0
1,E001,2025-06,151,118,33,0
2,E001,2025-07,159,140,19,0
3,E001,2025-08,103,68,35,0
4,E001,2025-09,99,64,35,0
...,...,...,...,...,...,...
1850,E199,2025-08,159,119,40,0
1851,E199,2025-09,161,119,42,1
1852,E199,2025-10,133,100,33,0
1853,E199,2025-11,126,88,38,0


## ✅ 1. Validación MÁS Importante

Se realiza la siguiente verificación:

asignaciones_mes.groupby(["id_empleado", "periodo"])["horas_cliente"].sum()

---

### 🔍 Resultado esperado

Valores como:

- 128  
- 136  
- 149  
- ...  

👉 Deben coincidir exactamente con **horas_trabajadas**

---

### 🔥 ¿Por qué es tan importante?

Porque valida la siguiente relación fundamental:

∑ horas_cliente = horas_trabajadas

---

### 💥 Implicaciones

Esto garantiza que:

- No se están creando horas artificialmente  
- No se están perdiendo horas  
- El modelo mantiene consistencia interna  

---

### 🧠 Conclusión

Esta validación confirma que la distribución de horas por cliente/proyecto está correctamente alineada con la actividad total registrada.

👉 Es un chequeo crítico para asegurar la calidad y confiabilidad del dataset.

In [25]:
# Objetivos

In [26]:
objetivos = []

for periodo in calendario["periodo"].dt.strftime("%Y-%m"):
    for region in empleados["region"].unique():

        objetivos.append({
            "periodo": periodo,
            "region": region,
            "objetivo_utilizacion": np.random.uniform(0.75, 0.85),
            "objetivo_costo": np.random.randint(40000, 80000)
        })

objetivos = pd.DataFrame(objetivos)

In [27]:
objetivos.head()

,periodo,region,objetivo_utilizacion,objetivo_costo
0,2025-01,Europe,0.826723,42024
1,2025-01,APAC,0.757512,41155
2,2025-01,LATAM,0.796549,58569
3,2025-01,North America,0.750488,42317
4,2025-02,Europe,0.770687,55916


## ✅ 1. Validación Final de Objetivos

Lo que muestra el modelo:

- periodo ✔  
- región ✔  
- objetivo_utilizacion (~0.75–0.85) ✔  
- objetivo_costo ✔  

👉 Todos los elementos tienen coherencia desde el punto de vista de negocio.

---

## 🧠 2. Modelo Construido (Resumen)

Se ha construido un modelo de datos tipo **estrella**, adecuado para análisis y reporting.

---

### 🧩 Dimensiones

- **empleados** → quién  
- **calendario** → cuándo  
- **región** → dónde (implícito)  

---

### 📊 Tablas de Hechos

1. **horas_trabajadas**  
   → actividad operativa  

2. **costos**  
   → impacto financiero  

3. **asignaciones_mes**  
   → distribución del trabajo por cliente/proyecto  

---

### 🎯 Tabla de Objetivos

- **objetivos**  
  → referencia para evaluación de KPIs  

---

## 🔥 3. Resolución del Problema

Objetivo del proyecto:

**Automatizar reportes de workforce**

---

### ✅ Capacidades actuales del modelo

Con la estructura construida ya es posible analizar:

- Utilización de empleados  
- Costos operativos  
- Distribución del trabajo por cliente  
- Comparación entre metas y resultados reales  

---

## 💥 Conclusión

El modelo ya permite construir un **dashboard completo de workforce analytics**, integrando:

- eficiencia operativa  
- impacto financiero  
- asignación de recursos  
- seguimiento de objetivos  

👉 Nivel listo para implementación en herramientas de BI (Power BI, dashboards, etc.).

In [28]:
empleados.to_csv("empleados.csv", index=False)
calendario.to_csv("calendario.csv", index=False)
horas_trabajadas.to_csv("horas_trabajadas.csv", index=False)
costos.to_csv("costos.csv", index=False)
asignaciones.to_csv("asignaciones.csv", index=False)
asignaciones_mes.to_csv("asignaciones_mes.csv", index=False)
objetivos.to_csv("objetivos.csv", index=False)

In [29]:
import os
os.listdir()

['.config',
 'asignaciones_mes.csv',
 'horas_trabajadas.csv',
 'calendario.csv',
 'asignaciones.csv',
 'costos.csv',
 'empleados.csv',
 'objetivos.csv',
 'sample_data']

In [30]:
from google.colab import files

files.download("empleados.csv")
files.download("horas_trabajadas.csv")
files.download("costos.csv")
files.download("asignaciones_mes.csv")
files.download("objetivos.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>